# 11 · ClearML pipeline + experiment tracking

Wire the PoC stages as ClearML Tasks. Run locally first (`start_locally`). Remote queues need a ClearML Agent, which this laptop compose stack does not start.

In [1]:
from cross_model_drift.config import load_config

config = load_config()
config.clearml_project, config.clearml_web_host, config.clearml_api_host

('cross-model-drift', 'http://localhost:8080', 'http://localhost:8008')

## Suggested tasks

`prepare_data` → `create_features` → `train_logistic_baseline` → `train_lightgbm` → `run_optuna_hpo` → `evaluate_model` → `compare_champion_challenger`

In [4]:
from clearml import PipelineController


def prepare_data(version: str, split: str) -> int:
    from cross_model_drift.data import load_split

    frame = load_split(version, split)  # type: ignore[arg-type]
    return len(frame)


def train_and_evaluate(version: str) -> dict[str, float]:
    from cross_model_drift.data import load_split
    from cross_model_drift.features import target_vector
    from cross_model_drift.metrics import quality_metrics
    from cross_model_drift.models import train_lightgbm

    train = load_split(version, "train")  # type: ignore[arg-type]
    valid = load_split(version, "validation")  # type: ignore[arg-type]
    test = load_split(version, "test")  # type: ignore[arg-type]
    model = train_lightgbm(train, target_vector(train), valid, target_vector(valid))
    return quality_metrics(target_vector(test), model.predict_proba(test))


pipe = PipelineController(
    name="fraud-champion-challenger",
    project=config.clearml_project,
    version="0.1.0",
    add_pipeline_tags=False,
)
pipe.add_function_step(
    name="prepare_v1_train",
    function=prepare_data,
    function_kwargs={"version": "v1", "split": "train"},
    function_return=["n_rows"],
)
pipe.add_function_step(
    name="train_v1",
    function=train_and_evaluate,
    function_kwargs={"version": "v1"},
    function_return=["v1_metrics"],
    parents=["prepare_v1_train"],
)
pipe.add_function_step(
    name="train_v2",
    function=train_and_evaluate,
    function_kwargs={"version": "v2"},
    function_return=["v2_metrics"],
    parents=["prepare_v1_train"],
)

# After ~/.clearml/clearml.conf exists:
pipe.start_locally(run_pipeline_steps_locally=True)
"pipeline defined; start locally once credentials exist"

Could not fetch function declared in __main__: <module '__main__'> is a built-in module
Could not fetch function imports: <module '__main__'> is a built-in module
Could not fetch function declared in __main__: <module '__main__'> is a built-in module
Could not fetch function imports: <module '__main__'> is a built-in module


ClearML Task: created new task id=3a6707536afa4fd1847df58c0e921a1e
ClearML results page: http://localhost:8080/projects/6ec3c4a0b4d34f89b09a543fa4f55236/tasks/3a6707536afa4fd1847df58c0e921a1e/output/log
ClearML pipeline page: http://localhost:8080/pipelines/6ec3c4a0b4d34f89b09a543fa4f55236/experiments/3a6707536afa4fd1847df58c0e921a1e


Could not fetch function declared in __main__: <module '__main__'> is a built-in module
Could not fetch function imports: <module '__main__'> is a built-in module


Launching the next 1 steps
Launching step [prepare_v1_train]
Launching step: prepare_v1_train
Parameters:
{'kwargs/version': 'v1', 'kwargs/split': 'train'}
Configurations:
{}
Overrides:
{}
ClearML results page: http://localhost:8080/projects/6ec3c4a0b4d34f89b09a543fa4f55236/tasks/dd61a8026c584771a00dd8f2f2efcbba/output/log
2026-08-23 20:22:27,784 - clearml.resource_monitor - WARNING - Could not fetch GPU stats: NVML Shared Library Not Found
ClearML Monitor: GPU monitoring failed getting GPU reading, switching off GPU monitoring
Launching the next 2 steps
Launching step [train_v2]
Launching step [train_v1]
Launching step: train_v1
Parameters:
{'kwargs/version': 'v1'}
Configurations:
{}
Overrides:
{}
Launching step: train_v2
Parameters:
{'kwargs/version': 'v2'}
Configurations:
{}
Overrides:
{}
ClearML results page: http://localhost:8080/projects/6ec3c4a0b4d34f89b09a543fa4f55236/tasks/aa896e0262384f5b8c00e3734ab7289f/output/log
ClearML results page: http://localhost:8080/projects/6ec3c4a0

'pipeline defined; start locally once credentials exist'

Open http://localhost:8080 after compose is up. Create API credentials, write `~/.clearml/clearml.conf` from `configs/clearml.conf.example`, then set `init=True` in earlier notebooks.